# Compare Prediction with validation data

Comparions map [here](https://terriamap.p.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FStasjonsdata+Bl%C3%B8tbunnsbasen%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22csv%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FStasjonsdata+Bl%C3%B8tbunnsbasen%22%2C%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%5D%2C%22timeline%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FStasjonsdata+Bl%C3%B8tbunnsbasen%22%2C%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A12.694702148437502%2C%22south%22%3A67.48754521444751%2C%22east%22%3A17.817077636718754%2C%22north%22%3A68.5111372120893%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Afalse%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D)

In [2]:
import pandas as pd
import geopandas as gpd

In [5]:
# Run once to setup points
pd.read_csv("./Stasjonsdata_bløtbunnsbasen 19.12.2025.csv").rename(
    columns={"y_coord_ny": "lat", "x_coord_ny": "lon"}
).drop_duplicates(subset=["lat", "lon"])[["lat", "lon", "DYP", "LOKALITET"]].to_csv(
    "./Stasjonsdata_blotbunnsbasen_points.csv", index=False
)

In [15]:
df = pd.read_csv("https://storage.googleapis.com/niva-geodata/MarintNaturKart/Stasjonsdata_blotbunnsbasen_points.csv")
gdf_blotbunn = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)


In [16]:
gdf_predict = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/nisjedata-substrat-xgbclassifier_norge_latest_25833.geo.parquet")

In [22]:
# Ensure both layers share the same CRS (reproject points to match polygons)
gdf_blotbunn_proj = gdf_blotbunn.to_crs(gdf_predict.crs)

# Filter polygons with desired BunnType
total_points = len(gdf_blotbunn_proj)
for bunn_type in gdf_predict["BunnType"].unique():
    soft_bottom_polygons = gdf_predict[gdf_predict["BunnType"] == bunn_type]

    # Spatial join: keep only points that fall inside a "løsbunn" polygon
    points_in_soft_bottom = gpd.sjoin(
        gdf_blotbunn_proj,
        soft_bottom_polygons[["BunnType", "geometry"]],
        how="inner",
        predicate="within",
    )   

    # Count how many points are inside bløtbunn polygons
    n_points_in_soft_bottom = len(points_in_soft_bottom)
    
    percent_in_soft_bottom = (n_points_in_soft_bottom / total_points) * 100

    print(f"Points in {bunn_type}: {n_points_in_soft_bottom}")
    print(f"Percentage in {bunn_type}: {percent_in_soft_bottom:.2f}% /n")

Points in løsbunn: 2068
Percentage in løsbunn: 86.17% /n
Points in fastbunn: 148
Percentage in fastbunn: 6.17% /n
Points in blanding: 37
Percentage in blanding: 1.54% /n
